<div style="padding:20px; border-radius:6px; background-color:#ffe6e6; color:#b30000; border-left:5px solid #b30000;">
<b>Jalankan notebook ini dengan environment <code>mandala_env</code>.</b><br/>
Siapkan <code>VAULT_TOKEN</code> (di environment atau file <code>.env</code>) untuk membaca kredensial S3 Mandala dari Vault.
</div>

# Tutorial: Akses Data INACAWO via Mandala

Marine Data Lake (**Mandala**) menyimpan observasi MMS1 dan dataset model (termasuk INACAWO)
di object storage. Di Mandalahub, Anda membaca data lewat protokol **S3** sebagai
**Zarr / zcollection** — tanpa mengunduh file NetCDF utuh ke workspace.

Notebook ini melanjutkan materi OPeNDAP: fokus living archive, catalog Mandala,
dan satu case study plot bergaya operasional OFS.


## 1. Apa itu Mandala / Mandalahub / Zarr?

| Konsep | Arti singkat |
|---|---|
| **Data lake** | Penyimpanan fleksibel; data masuk tanpa wajib transform penuh dulu |
| **Mandala** | Data lake maritim BMKG (observasi + NWP) di S3 on-prem |
| **Zarr / zcollection** | Format chunked; baca hanya potongan yang dibutuhkan, cocok object storage |
| **Mandalahub** | JupyterHub analisis di jaringan lokal ke S3 Mandala |

Untuk INACAWO di Mandala: arsip **living FULL** cycle **00Z & 12Z** — ditumpuk / di-override
tiap cycle terbaru (bukan snapshot beku per run). Hindcast multi-dekade juga tersedia
lewat catalog `model_hindcast`.

**Bedakan dari OPeNDAP:** OPeNDAP = file NetCDF forecast retensi pendek + subset remote;
Mandala = arsip panjang + I/O paralel Zarr di Mandalahub.


## 2. Struktur data Mandala vs OPeNDAP

Keduanya berasal dari postproc ilmiah INACAWO yang sama, tetapi **kemasan berbeda**.
Selalu cek `.attrs` sebelum membandingkan angka antar sumber.

### Dimensi & naming
| | OPeNDAP | Mandala (contoh) |
|---|---|---|
| Waktu | `time` | `TSTEP` |
| Vertikal | `depth` (Flows) / — | `LAY` (layer; di WRF = level tekanan) |
| Spatials | `lat`, `lon` | `ROW`, `COL` (+ coord `lat`/`lon`) |
| SWH | `hs` | `sig_wave_height` |
| Wind U/V | `uwnd` / `vwnd` | `U_WIND` / `V_WIND` (`inacawo_wrf_3d`) |
| Current | `u` / `v` | `ubar_eastward` (+ pasangan V bila ada) |
| SSH | `zeta` | `zeta` |

### Collection yang dipakai di tutorial
| Path collection | Isi tipikal |
|---|---|
| `mandala-prod1/datalake/model/collections/inacawo_ocean_2d` | ocean/wave 2D surface fields |
| `mandala-prod1/datalake/model/collections/inacawo_wrf_3d` | atmosfer 3D |

### Satuan & arah (ingat!)
- Angin OPeNDAP Atmos sering **m/s**; file Wave OPeNDAP bisa **knots**.
- Plot OFS menampilkan angin dalam **knot** (`× 1.94384` dari m/s).
- Arus OPeNDAP biasanya **cm/s**; di Mandala/plotter kadang m/s lalu `×100` — **cek attrs + magnitudo**.
- Komponen `*_eastward` / `*_northward` = vektor *ke arah* (CF).
- Field `*_dir`: pastikan konvensi from/to; helper `plot_ofs_map` mengikuti **ofs-plotter**.


## 3. Import & koneksi S3 via Vault

Jangan hard-code Access Key / Secret Key di notebook.
Token Vault bisa di-set di environment atau di cell di bawah (jangan commit token).


In [ ]:
import os
from datetime import datetime
from pathlib import Path
from typing import Literal

import fsspec
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
import zcollection as zc
import zcollection.fs_utils as zc_fs
import mandala

from training_ofs import (
    load_env_file,
    load_s3_options_from_vault,
    plot_ofs_map,
    list_params,
)

# Optional: load VAULT_TOKEN from .env
ENV_PATH = Path.cwd() / ".env"
if not ENV_PATH.is_file():
    ENV_PATH = Path("/home/jupyter-tyo/training-ofs-2026/.env")
load_env_file(ENV_PATH)

# Paste token here only for local runs if not already in the environment.
VAULT_TOKEN = os.environ.get("VAULT_TOKEN", "")

s3_options = load_s3_options_from_vault(token=VAULT_TOKEN or None)

fs = fsspec.filesystem(
    "s3",
    key=s3_options["key"],
    secret=s3_options["secret"],
    client_kwargs=s3_options["client_kwargs"],
)
fs


## 4. Catalog Mandala

Catalog memetakan dataset observasi / model / hindcast.
- `obs` → observasi MMS1
- `model` → living forecast archive (termasuk INACAWO operasional)
- `model_hindcast` → hindcast (retensi panjang; hati-hati load tanpa filter)


In [ ]:
def get_catalog(
    database: Literal["obs", "model", "model_hindcast"],
    s3_options: dict,
):
    catalog = {
        "obs": "catalog.json",
        "model": "catalog.json",
        "model_hindcast": "catalog_hindcast.json",
    }.get(database, "catalog.json")
    if database == "model_hindcast":
        database = "model"
    catalog_path = os.path.join("mandala-prod1/datalake/", database, catalog)
    return mandala.MandalaCatalog(
        path=catalog_path,
        protocol="s3",
        options=s3_options,
    )


In [ ]:
catalogobs = get_catalog("obs", s3_options)
catalogobs


In [ ]:
catalogmodel = get_catalog("model", s3_options)
catalogmodel


In [ ]:
catalogmodelhindcast = get_catalog("model_hindcast", s3_options)
catalogmodelhindcast


## 5. Membaca collection INACAWO dengan zcollection

Semua dataset disimpan sebagai **zcollection** (di atas Zarr), efisien diproses dengan `xarray`.

**Wajib:** pakai `filters` tanggal sempit. Load tanpa filter bisa menarik ratusan partition.


In [ ]:
fs.invalidate_cache()
ocean2d_collection = zc.open_collection(
    path="mandala-prod1/datalake/model/collections/inacawo_ocean_2d",
    mode="r",
    filesystem=zc_fs.get_fs(filesystem=fs),
)

# Jaga window kecil agar ramah sistem (sesuaikan jika partition berubah)
date_filter = (
    "(year == 2026 and month == 9 and day >= 12) or "
    "(year == 2026 and month == 9 and day <= 13)"
)

dset = ocean2d_collection.load(delayed=True, filters=date_filter).to_xarray()
dset


### Inspect atribut (latihan)

Bandingkan nama variabel Mandala dengan short name OPeNDAP dari tutorial sebelumnya.


In [ ]:
def summarize_mandala(ds: xr.Dataset) -> pd.DataFrame:
    rows = []
    for name, da in ds.data_vars.items():
        rows.append(
            {
                "variable": name,
                "dims": tuple(da.dims),
                "units": da.attrs.get("units"),
                "long_name": da.attrs.get("long_name") or da.attrs.get("var_desc"),
                "standard_name": da.attrs.get("standard_name"),
            }
        )
    return pd.DataFrame(rows).sort_values("variable")

summarize_mandala(dset).head(20)


In [ ]:
# Quicklook bawaan xarray (bukan layout OFS)
dset.isel(TSTEP=10).sig_wave_height.plot()
plt.title("sig_wave_height quicklook (full domain — hanya 1 timestep)")
plt.show()


## 6. Case study: SWH Laut Jawa dari living archive

**Skenario.** Ambil `sig_wave_height` untuk window singkat, crop ke Laut Jawa,
lalu plot dengan **layout OFS** (`param='swh'`) — sama gaya produk operasional.

Crop spatial **setelah** load bertarget filter tanggal, sebelum `plot_ofs_map`.


In [ ]:
# Ambil 1 timestep di tengah window, crop Laut Jawa
sample = dset["sig_wave_height"].isel(TSTEP=min(10, dset.sizes["TSTEP"] - 1))
# ROW/COL are lat/lon coordinates on this collection
sample = sample.sel(ROW=slice(-7.0, -3.0), COL=slice(106.0, 115.0))

# Flatten LAY if present
if "LAY" in sample.dims:
    sample = sample.isel(LAY=0)

lat = sample["ROW"].values if "ROW" in sample.coords else sample["lat"].values
lon = sample["COL"].values if "COL" in sample.coords else sample["lon"].values
field = np.squeeze(sample.values)

# Timestamp for OFS header
tstamp = pd.to_datetime(sample["TSTEP"].values) if "TSTEP" in sample.coords else None
if tstamp is None and "TSTEP" in dset.coords:
    tstamp = pd.to_datetime(dset["TSTEP"].values[min(10, dset.sizes["TSTEP"] - 1)])
baserun = pd.Timestamp(tstamp).to_pydatetime() if tstamp is not None else datetime.utcnow()

# Optional direction if available on same timestep
direction = None
for cand in ("sig_wave_dir", "mean_wave_dir", "wave_dir", "dir"):
    if cand in dset:
        d = dset[cand].isel(TSTEP=min(10, dset.sizes["TSTEP"] - 1))
        if "LAY" in d.dims:
            d = d.isel(LAY=0)
        d = d.sel(ROW=slice(-7.0, -3.0), COL=slice(106.0, 115.0))
        direction = np.squeeze(d.values)
        break

fig, ax = plot_ofs_map(
    lon,
    lat,
    field,
    param="swh",
    direction=direction,
    baserun=baserun,
    forecast=baserun,
    wind_input="INACAWO",
    map_area="Laut Jawa",
    quiver_skip=6,
)
plt.show()


### Interpretasi singkat

Catat tinggi gelombang signifikan di Laut Jawa pada waktu sample:
apakah kondisi relatif tenang / sedang untuk pelayaran umum? Bandingkan mental
dengan produk OPeNDAP cycle yang sama (jika masih dalam retensi Hyrax).


## 7. Bonus: atmos 3D (`inacawo_wrf_3d`)

Pola akses sama — ganti path collection. Tetap filter tanggal sempit.


In [ ]:
fs.invalidate_cache()
wrf3d_collection = zc.open_collection(
    path="mandala-prod1/datalake/model/collections/inacawo_wrf_3d",
    mode="r",
    filesystem=zc_fs.get_fs(filesystem=fs),
)

date_filter_wrf = (
    "(year == 2026 and month == 9 and day >= 12) or "
    "(year == 2026 and month == 9 and day <= 13)"
)

dset_wrf = wrf3d_collection.load(delayed=True, filters=date_filter_wrf).to_xarray()
dset_wrf


## 8. Best practice & catatan advanced

**Lakukan**
- Baca langsung S3 + Zarr di Mandalahub (jaringan lokal)
- Filter tanggal / pilih variabel sebelum compute penuh
- Cek `units` & naming sebelum bandingkan dengan OPeNDAP
- Plot case study pakai `plot_ofs_map` (style `ofs-plotter`)

**Hindari**
- Unduh file OPeNDAP penuh lalu upload ke Mandalahub
- Load hindcast multi-tahun tanpa filter
- Hard-code secret S3 di notebook / git

### Tip hindcast `TFLAG`
Pada beberapa collection hindcast, `TFLAG` punya ukuran `VAR` yang berubah antar era.
Jika `load()` gagal dengan *conflicting dimensions*, exclude `TFLAG` via
`selected_variables=[...]` dan tetap filter tanggal sempit.


## 9. Ringkasan pintu akses

| Kebutuhan | Pintu |
|---|---|
| Subset remote cepat / luar Mandalahub | OPeNDAP Hyrax |
| Analisis deret waktu / domain luas di Mandalahub | Mandala S3 + Zarr |
| Plot selaras produk operasional | `training_ofs.plot_ofs_map` (param dari ofs-plotter) |

Parameter plot tersedia: lihat `list_params()` di bawah.


In [ ]:
print("OFS plot params:", ", ".join(list_params()))
